In [ ]:
import os
import re
import glob
import math
import numpy as np
import matplotlib as plt
from scipy.io import loadmat
from scipy.stats import linregress,chi2,ttest_1samp
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from util import fisherztrans,stat_m_e

# Define screen parameters
screen_height_cm = 50
screen_height = 1080
screen_width = 1920
screen_distance_cm = 160
length_per_pixel_cm = screen_height_cm / screen_height

# subj
subjs = ['CN040','CN041','CN042','CN043','CN044','CN045','CN055','CN056']
tasks = ['face','fixation']

# ========= The raw data is available upon reasonable request. ========
#data_folderpath = 'C:/Users/Lenovo/Desktop/eye/eye_data'
#result_folderpath = 'C:/Users/Lenovo/Desktop/eye/eye_results'

result_folderpath = 'eye_results/'


In [ ]:
# data convertion: edf2asc
def edf2asc(session_folders):
    for session_folder in session_folders:
        filenames = glob.glob(session_folder+'/*.edf')
        for filename in filenames:
            ! edf2asc $filename -y -input


# return sessions and runs for each subject
def sess_info(subj,data_folderpath):

    # Path to the folder containing all session subfolders
    if subj == 'CN056':
        sessions = [1,2,3,4,5,6,7,8,9]
    elif subj == 'CN055':
        sessions = [3,4,5,6,7,8]
    else:
        sessions = [1,2,3,4,5,6,7,8]
    session_folderpath = data_folderpath+'/'+subj+'/'
    session_folders = []
    for sess in sessions:
        session_folders.append(glob.glob(session_folderpath+'*session'+str(sess))[0]+'/')

    # run list for all the sessions
    if subj == 'CN040':
        run_lists = {'face':[],'fixation':[]}
        sessionnum_lists = {'face':[['']*5]*8,'fixation':[['']*5]*8}
        for sess in sessions:
            if sess == 1:
                run_lists['face'].append([5,7,9])
                run_lists['fixation'].append([4,6,8,10])
            else:
                run_lists['face'].append([1,3,5,7,9])
                run_lists['fixation'].append([2,4,6,8,10])

    elif (subj=='CN041') | (subj=='CN042') | (subj=='CN045') | (subj=='CN055'):
        run_lists = {'face':[],'fixation':[]}
        sessionnum_lists = {'face':[['']*5]*8,'fixation':[['']*5]*8}
        for sess in sessions:
            run_lists['face'].append([1,3,5,7,9])
            run_lists['fixation'].append([2,4,6,8,10])

    elif subj == 'CN043':
        run_lists = {'face':[],'fixation':[]}
        sessionnum_lists = {'face':[['']*5]*8,'fixation':[['']*5]*8}
        for sess in sessions:
            if sess == 1:
                run_lists['face'].append([3,5,7,9])
                run_lists['fixation'].append([2,4,6,8,10])
            else:
                run_lists['face'].append([1,3,5,7,9])
                run_lists['fixation'].append([2,4,6,8,10]) 

    elif subj == 'CN044':
        run_lists = {'face':[],'fixation':[]}
        sessionnum_lists = {'face':[],'fixation':[]}
        for sess in sessions:
            if sess == 1:
                run_lists['face'].append([7,9])
                run_lists['fixation'].append([2,4,6,8,10])
                sessionnum_lists['face'].append([1,1])
                sessionnum_lists['fixation'].append([1]*5)
            elif sess == 7:
                run_lists['face'].append([1,3,5,7,9,1,3])
                run_lists['fixation'].append([2,4,6,8,10])
                sessionnum_lists['face'].append([7]*5+[1]*2)
                sessionnum_lists['fixation'].append([7]*5)
            elif sess == 8:
                run_lists['face'].append([1,3,5,7,9,5])
                run_lists['fixation'].append([2,4,6,8,10])
                sessionnum_lists['face'].append([8]*5+[1])
                sessionnum_lists['fixation'].append([8]*5)        
            else:
                run_lists['face'].append([1,3,5,7,9])
                run_lists['fixation'].append([2,4,6,8,10])      
                sessionnum_lists['face'].append(['']*5)
                sessionnum_lists['fixation'].append(['']*5)

    elif subj == 'CN056':
        run_lists = {'face':[],'fixation':[]}
        sessionnum_lists = {'face':[],'fixation':[]}
        for sess in sessions:
            if sess == 1:
                run_lists['face'].append([1,3,5])
                run_lists['fixation'].append([2,4,6])
                sessionnum_lists['face'].append([1]*3)
                sessionnum_lists['fixation'].append([1]*3)
            elif sess == 5:
                run_lists['face'].append([9])
                run_lists['fixation'].append([8,10])
                sessionnum_lists['face'].append([5])
                sessionnum_lists['fixation'].append([5,5])        
            elif sess == 6:
                run_lists['face'].append([1,3,5,7,9,9])
                run_lists['fixation'].append([2,4,6,8,10,10])
                sessionnum_lists['face'].append([6]*5+[1])
                sessionnum_lists['fixation'].append([6]*5+[1])    
            elif sess == 8:
                run_lists['face'].append([1,3,5,7,9,7])
                run_lists['fixation'].append([2,4,6,8,10,8])
                sessionnum_lists['face'].append([8]*5+[1])
                sessionnum_lists['fixation'].append([8]*5+[1])    
            elif sess == 9:
                run_lists['face'].append([1,3,5,7])
                run_lists['fixation'].append([2,4,6])
                sessionnum_lists['face'].append([5]*4)
                sessionnum_lists['fixation'].append([5]*3)                        
            else:
                run_lists['face'].append([1,3,5,7,9])
                run_lists['fixation'].append([2,4,6,8,10])      
                sessionnum_lists['face'].append(['']*5)
                sessionnum_lists['fixation'].append(['']*5)
    return sessions,session_folders,run_lists,sessionnum_lists


# Smoothing function
def smooth_data(data, sampling_rate, window_length_ms=50):
    window_length_samples = int(sampling_rate / 1000 * window_length_ms)
    if window_length_samples > data.shape[0]:
        window_length_samples = data.shape[0]
    smth_kernel = np.ones(window_length_samples) / window_length_samples 
    smoothed_data = np.zeros(data.shape)
    for i in range(data.shape[1]):
        smoothed_data[:, i] = np.convolve(data[:, i], smth_kernel, mode='same')
    return smoothed_data

# Function to process each session
def process_session(session_folder, run_list, sessionnum_list):
    '''
    session_folder: folder path of the session. e.g., ../CN040/session2
    run_list: list of runs for this session for a task. e.g., [1,3,5,7,9]
    sessionnum_list: list of experimental session numbers which may be different in this scanning session. 
                    all-empty means there is no run for difference session in this scanning.
                    e.g., [7,7,7,7,1,1], indicating that the besides 5 runs for session7, 2 runs 
                    for session1 were additionally acquired in this scanning session.
    '''

    preprocessed_data = {}
    deleted_trials = []

    # Process each file
    for run_i in range(len(run_list)):
        run = run_list[run_i]
        ascfile = glob.glob(session_folder+'*session'+str(sessionnum_list[run_i])+'*run'+str(run)+'*.asc')[0]
        matfile = glob.glob(session_folder+'*session'+str(sessionnum_list[run_i])+'*run'+str(run)+'*.mat')[0]
        
        #if f'run{run_i}' not in preprocessed_data:
        preprocessed_data[f'run{run_i}'] = {}
        preprocessed_data[f'run{run_i}']['deleted_run'] = []

        trial_data = {}
        with open(ascfile, 'r') as file:
            current_trial = None
            line_index = 0
            for line in file:
                if 'trial_' in line:
                    current_trial = line.strip()
                    trial_data[current_trial] = []
                    line_index = 0
                    blink_indices = set()
                elif current_trial:
                    elements = line.strip().split()
                    # delete blink data
                    if len(elements) >= 5 and all(element.replace('.', '', 1).isdigit() or element == '.' for element in elements[:5]):
                        numeric_elements = [float(element) if element != '.' else None for element in elements[:5]]
                        if numeric_elements[1] == None and numeric_elements[2] == None:
                            blink_indices.update(range(max(0, line_index - 50), line_index + 75))  
                        if line_index not in blink_indices:  
                            trial_data[current_trial].append(numeric_elements)
                        line_index += 1
                        if (line_index > 250) & ('trial_432' in current_trial):
                            break

            run_data_x = np.array([]) 
            run_data_y = np.array([])
            trial_ind = np.array([]) 
            for trial, samples in trial_data.items():
                valid_samples = [sample for sample in samples if sample[1] is not None and sample[2] is not None]
                if len(valid_samples) > 0:
                    run_data_x = np.append(run_data_x, np.array(valid_samples)[:,1])
                    run_data_y = np.append(run_data_y, np.array(valid_samples)[:,2])
                    trial_ind = np.append(trial_ind, np.ones(len(valid_samples))*(int(re.search(r'\d+$', trial).group())-1))
                else:
                    #print(f"No valid data for trial {trial}, skipping.")
                    deleted_trials.append(trial)
            if len(run_data_x) == 0:
                continue

            run_data_raw = np.vstack([run_data_x,run_data_y]).T
            preprocessed_data[f'run{run_i}']['run_data_raw'] = run_data_raw
            preprocessed_data[f'run{run_i}']['trial_ind'] = trial_ind.astype(int)

            #  transform pixel to degree
            unit = math.atan(1 * length_per_pixel_cm / screen_distance_cm) * 180 / np.pi
            # coordinate centering
            degree_x = (run_data_raw[:,0] - screen_width / 2) * unit
            degree_y = (screen_height - run_data_raw[:,1] - screen_height / 2) * unit
            degree_data=np.vstack([degree_x, degree_y]).T
            # excluded samples deviating more than 6° from central fixation, excising data 250 ms before and 250 ms after each occurrence
            indices = np.union1d(np.where(abs(degree_x)>6)[0],np.where(abs(degree_y)>6)[0])
            outliers_indices = set()
            for ind in indices:
                outliers_indices.update(range(max(0,ind-125), min(ind+125,len(degree_data))))
            degree_data = np.delete(degree_data,list(outliers_indices),0)
            trial_ind = np.delete(trial_ind,list(outliers_indices),0)
            preprocessed_data[f'run{run_i}']['degree_data'] = degree_data
            preprocessed_data[f'run{run_i}']['trial_ind'] = trial_ind.astype(int)

            if len(degree_data)>0:
                # delete drift
                slope_x, intercept_x, _, _, _ = linregress(range(degree_data.shape[0]), list(degree_data[:,0]))
                slope_y, intercept_y, _, _, _ = linregress(range(degree_data.shape[0]), list(degree_data[:,1]))

                corrected_x = degree_data[:,0] - (slope_x * np.arange(0,degree_data.shape[0]) + intercept_x)
                corrected_y = degree_data[:,1] - (slope_y * np.arange(0,degree_data.shape[0]) + intercept_y)
                drift_data = np.vstack([corrected_x, corrected_y]).T
                preprocessed_data[f'run{run_i}']['drift_data'] = drift_data


                # centered_data
                detrend_centered_data = drift_data - np.array([np.median(drift_data[:,0]),np.median(drift_data[:,1])])
                preprocessed_data[f'run{run_i}']['detrend_centered_data'] = detrend_centered_data        

                # downsampling
                downsampling_data = detrend_centered_data[::5,:]  # 500Hz -> 100Hz
                trial_ind = trial_ind[::5]  # 500Hz -> 100Hz
                preprocessed_data[f'run{run_i}']['downsampling_data'] = downsampling_data       
                preprocessed_data[f'run{run_i}']['trial_ind'] = trial_ind.astype(int)

                # smooth
                sampling_rate = 100
                smoothing_data = smooth_data(downsampling_data, sampling_rate)
                preprocessed_data[f'run{run_i}']['smoothing_data'] = smoothing_data

            '''
            # exclusion
            if len(preprocessed_data[f'run{run_i}']['trial_ind'])/sampling_rate/432 < 1/3:
                preprocessed_data[f'run{run_i}']['smoothing_data'] = np.array([])
                preprocessed_data[f'run{run_i}']['deleted_run'].append(run)
                print(f'run{run} removed : {len(preprocessed_data[f'run{run_i}']['trial_ind'])/sampling_rate/432.:2f}')
            '''

        
        # align with flag data
        all_samples_with_flags = []
        mat_data = loadmat(matfile)

        if 'this_posi_list' in mat_data:
            matrix = mat_data['this_posi_list']
            flattened_matrix = matrix.flatten()
        else:
            matrix = mat_data['result'][0][0]['this_posi_list']
            flattened_matrix = matrix.flatten()
        for run_key, run_data in preprocessed_data.items():
            if ('smoothing_data' in run_data):
                smoothing_data = run_data['smoothing_data']
                if len(smoothing_data)>0:
                    trial_ind = run_data['trial_ind']
                    trial_flag = flattened_matrix[trial_ind]
                    for sample in zip(smoothing_data[:, 0], smoothing_data[:, 1], trial_flag):
                        trial_num = re.search(r'\d+$', str(sample[-1]))
                        if trial_num and trial_num.group() not in deleted_trials:
                            all_samples_with_flags.append(sample)
    return all_samples_with_flags


# 2d_gaussian
def gaussian_2d_with_rotation(xy, A, x0, y0, sigma_x, sigma_y, theta):
    x, y = xy
    x_rot = (x - x0) * np.cos(theta) - (y - y0) * np.sin(theta)
    y_rot = (x - x0) * np.sin(theta) + (y - y0) * np.cos(theta)
    g = A * np.exp(-(x_rot)**2 / (2 * sigma_x**2) - (y_rot)**2 / (2 * sigma_y**2))
    return g.flatten()  

# heatmap
def heatmap2d(x,y):
    heatmap, xedges, yedges = np.histogram2d(x, y, bins=100, range=[[-5, 5], [-5, 5]], density=True)
    x_center = 0.5 * (xedges[1:] + xedges[:-1])
    y_center = 0.5 * (yedges[1:] + yedges[:-1])

    xg, yg = np.meshgrid(x_center, y_center)
    return xg, yg, heatmap, xedges, yedges

# 2d_gaussian_fitting 
def fitting_gaussian(x,y):
    xg,yg,heatmap,xedges,yedges = heatmap2d(x,y)
    # fitting with constraints
    try:
        popt, pcov = curve_fit(
            gaussian_2d_with_rotation, (xg, yg), heatmap.flatten(),
            p0=[1, np.mean(xg), np.mean(yg), np.std(xg), np.std(yg), 0],
            bounds=([0, -5, -5, 0, 0, -np.pi/2],  
                [np.inf, 5, 5, 5, 5, np.pi/2]))
    except (RuntimeError, ValueError) as e:
        print(f"Fit error:{e}")
        popt = np.nan
    return popt

# calulate 95% contour areas
def cal_areas(popt):
    chi = chi2.ppf(0.95,df=2)
    A, x0, y0, sigma_x, sigma_y, theta = popt
    major_axis = np.sqrt(chi) * sigma_x
    minor_axis = np.sqrt(chi) * sigma_y
    area = np.pi * np.abs(major_axis) * np.abs(minor_axis)
    return area

# plot heatmap
def plot_heatmap_with_contour(ax, x, y, popt, title='', ifplotarea=0):
    xg,yg,heatmap,xedges,yedges = heatmap2d(x,y)
    
    # predicting
    A, x0, y0, sigma_x, sigma_y, theta = popt
    x_rot = (xg - x0) * np.cos(theta) - (yg - y0) * np.sin(theta)
    y_rot = (xg - x0) * np.sin(theta) + (yg - y0) * np.cos(theta)
    g = A * np.exp(-(x_rot)**2 / (2 * sigma_x**2) - (y_rot)**2 / (2 * sigma_y**2))
    gaussian_values = g.flatten()
    gaussian_values = gaussian_values.reshape(100, 100)

    # plot heatmap
    ax.grid(True, which='both', color='gray', linestyle='--', linewidth=1)
    ax.set_xticks(np.arange(-5, 5.1, step=2))
    ax.set_yticks(np.arange(-5, 5.1, step=2))
    ax.imshow(heatmap, extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], origin='lower', cmap='hot')

    chi = chi2.ppf(0.95,df=2)
    threshold = A * np.exp(-chi/2)
    contour = ax.contour(xg, yg, gaussian_values, levels=[threshold], colors='lightsteelblue', linewidths=2)

    # plot 95% contour
    if len(contour.collections[0].get_paths()) > 0:
        contour_points = contour.collections[0].get_paths()[0].vertices
        if ifplotarea:
            area = cal_areas(popt)
            ax.set_title(title + ' - Area: {:.2f}'.format(area))
    else:
        ax.set_title(title + ' - No contours found')        



    ax.set_xticklabels([])
    ax.set_yticklabels([])


### pool data across sessions

In [ ]:
# =========================================== #
# Set processing procedures
ifdataconvert = 0
ifpreprocessing = 1
iffitting = 1
ifcalarea = 1
ifplotting = 1
ifplotarea = 0
# =========================================== #


for subj_i in range(1,len(subjs)):#range(len(subjs)):
    subj = subjs[subj_i]
    sessions,session_folders,run_lists,sessionnum_lists = sess_info(subj,data_folderpath)

    # edf2asc
    if ifdataconvert:
        edf2asc(session_folders)

    # preprocessing
    if ifpreprocessing:
        # combining all data
        all_sessions_data = {'face':[],'fixation':[]}
        sep_sessions_data = {'face':[],'fixation':[]}
        all_classified_data = {'face':{},'fixation':{}}
        sep_classified_data = {'face':{},'fixation':{}}
        for task in tasks:
            for s in range(len(sessions)):
                sess = sessions[s]
                run_list = run_lists[task][s]
                sessionnum_list = sessionnum_lists[task][s]
                session_folder = session_folders[s]
                print(f'{session_folder}\t{run_list}')
                processed_data = process_session(session_folder, run_list, sessionnum_list)
                all_sessions_data[task].extend(processed_data)
                sep_sessions_data[task].append(processed_data)
                
                sep_classified_data[task]['s'+str(sess)] = {}
                for sample in processed_data:
                    flag = sample[-1]
                    if flag not in sep_classified_data[task]['s'+str(sess)]:
                        sep_classified_data[task]['s'+str(sess)][flag] = []
                    sep_classified_data[task]['s'+str(sess)][flag].append(sample)
                
            for sample in all_sessions_data[task]:
                flag = sample[-1]
                if flag not in all_classified_data[task]:
                    all_classified_data[task][flag] = []
                all_classified_data[task][flag].append(sample)

    # gaussian fit
    if os.path.exists((f'{result_folderpath}/fitting_params.npz')):
        all_popt = np.load(f'{result_folderpath}/fitting_params.npz',allow_pickle = True)['all_popt']
    else:
        all_popt = np.zeros([8,2,16,6])*np.nan # 8subj x 2task x 16position x 6param
    if iffitting:
        for task_i in range(len(tasks)):
            task = tasks[task_i]
            sessions_data = all_classified_data[task]
            for flag in range(16):
                data = np.array(sessions_data[flag])    
                if data.size > 0:
                    x, y = data[:, 0], data[:, 1]
                    all_popt[subj_i,task_i,flag,:] = fitting_gaussian(x, y)
        np.savez(f'{result_folderpath}/fitting_params.npz',all_popt=all_popt)

    # calculate 95% contour areas
    if os.path.exists((f'{result_folderpath}/contour_areas.npz')):
        all_area = np.load(f'{result_folderpath}/contour_areas.npz',allow_pickle = True)['all_area']
    else:
        all_area = np.zeros([8,2,16])*np.nan # 8subj x 2task x 16position
    if ifcalarea:
        for task_i in range(len(tasks)):
            task = tasks[task_i]
            sessions_data = all_classified_data[task]
            # Generate plot for the session
            for flag in range(16):
                data = np.array(sessions_data[flag])
                popt = all_popt[subj_i,task_i,flag,:]
                if data.size > 0:
                    x, y = data[:, 0], data[:, 1]
                    all_area[subj_i,task_i,flag] = cal_areas(popt)
        np.savez(f'{result_folderpath}/contour_areas.npz', all_area=all_area)

    # plot data for each subjects
    if ifplotting:
        for task_i in range(len(tasks)):
            task = tasks[task_i]
            sessions_data = all_classified_data[task]
            # Generate plot for the session
            fig, axes = plt.subplots(4, 4, figsize=(8, 8))
            for flag, ax in enumerate(axes.ravel(), 1):
                data = np.array(sessions_data[flag-1])
                popt = all_popt[subj_i,task_i,flag-1,:]
                if data.size > 0:
                    x, y = data[:, 0], data[:, 1]
                    plot_heatmap_with_contour(ax, x, y, popt, title='', ifplotarea=ifplotarea)

            plt.tight_layout()
            plt.savefig(f'{result_folderpath}/{task}_{subj}.png')

     

### pool data across runs for each session

In [ ]:
# =========================================== #
# Set processing procedures
ifdataconvert = 0
ifpreprocessing = 1
iffitting = 1
ifcalarea = 1
ifplotting = 0
ifplotarea = 0
# =========================================== #


for subj_i in range(7,len(subjs)):#range(len(subjs)):
    subj = subjs[subj_i]
    sessions,session_folders,run_lists,sessionnum_lists = sess_info(subj,data_folderpath)

    # edf2asc
    if ifdataconvert:
        edf2asc(session_folders)

    # preprocessing
    if ifpreprocessing:
        # combining data for each session
        all_sessions_data = {'face':[],'fixation':[]}
        sep_sessions_data = {'face':[],'fixation':[]}
        all_classified_data = {'face':{},'fixation':{}}
        sep_classified_data = {'face':{},'fixation':{}}
        for task in tasks:
            for s in range(len(sessions)):
                sess = sessions[s]
                run_list = run_lists[task][s]
                sessionnum_list = sessionnum_lists[task][s]
                session_folder = session_folders[s]
                print(f'{session_folder}\t{run_list}')
                processed_data = process_session(session_folder, run_list, sessionnum_list)
                all_sessions_data[task].extend(processed_data)
                sep_sessions_data[task].append(processed_data)
                
                sep_classified_data[task]['s'+str(sess)] = {}
                for sample in processed_data:
                    flag = sample[-1]
                    if flag not in sep_classified_data[task]['s'+str(sess)]:
                        sep_classified_data[task]['s'+str(sess)][flag] = []
                    sep_classified_data[task]['s'+str(sess)][flag].append(sample)
                
            for sample in all_sessions_data[task]:
                flag = sample[-1]
                if flag not in all_classified_data[task]:
                    all_classified_data[task][flag] = []
                all_classified_data[task][flag].append(sample)

    # gaussian fit
    if os.path.exists((f'{result_folderpath}/fitting_params_session.npz')):
        all_popt = np.load(f'{result_folderpath}/fitting_params_session.npz',allow_pickle = True)['all_popt']
    else:
        all_popt = np.zeros([8,2,9,16,6])*np.nan # 8subj x 2task x 9 session x 16position x 6param
    if iffitting:
        for task_i in range(len(tasks)):
            task = tasks[task_i]
            for s in range(len(sessions)):
                sess = sessions[s]
                sessions_data = sep_classified_data[task]['s'+str(sess)]
                for flag in range(16):
                    if flag in sessions_data.keys():
                        data = np.array(sessions_data[flag])    
                        if data.size > 0:
                            x, y = data[:, 0], data[:, 1]
                            all_popt[subj_i,task_i,s,flag,:] = fitting_gaussian(x, y)
        np.savez(f'{result_folderpath}/fitting_params_session.npz',all_popt=all_popt)

    # calculate 95% contour areas
    if os.path.exists((f'{result_folderpath}/contour_areas_session.npz')):
        all_area = np.load(f'{result_folderpath}/contour_areas_session.npz',allow_pickle = True)['all_area']
    else:
        all_area = np.zeros([8,2,9,16])*np.nan # 8subj x 2task x 9session x 16position
    if ifcalarea:
        for task_i in range(len(tasks)):
            task = tasks[task_i]
            for s in range(len(sessions)):
                sess = sessions[s]
                sessions_data = sep_classified_data[task]['s'+str(sess)]
                # Generate plot for the session
                for flag in range(16):
                    if flag in sessions_data.keys():                    
                        data = np.array(sessions_data[flag])
                        popt = all_popt[subj_i,task_i,s,flag,:]
                        if data.size > 0:
                            x, y = data[:, 0], data[:, 1]
                            all_area[subj_i,task_i,s,flag] = cal_areas(popt)
        np.savez(f'{result_folderpath}/contour_areas_session.npz', all_area=all_area)

    # plot data for each subjects
    if ifplotting:
        for task_i in range(len(tasks)):
            task = tasks[task_i]
            for s in range(len(sessions)):
                sess = sessions[s]
                sessions_data = sep_classified_data[task]['s'+str(sess)]            
                # Generate plot for the session
                fig, axes = plt.subplots(4, 4, figsize=(8, 8))
                for flag, ax in enumerate(axes.ravel(), 1):
                    if flag in sessions_data.keys():                    
                        data = np.array(sessions_data[flag-1])
                        popt = all_popt[subj_i,task_i,flag-1,:]
                        if data.size > 0:
                            x, y = data[:, 0], data[:, 1]
                            plot_heatmap_with_contour(ax, x, y, popt, title='', ifplotarea=ifplotarea)

                plt.tight_layout()
                plt.savefig(f'{result_folderpath}/{task}_{subj}_session'+sess+'.png')



### correlation

In [ ]:
ifcalcorr = 1
# calcuate correlation
if os.path.exists((f'{result_folderpath}/stim_eye_corr.npz')):
    all_corr = np.load(f'{result_folderpath}/stim_eye_corr.npz',allow_pickle = True)['all_corr']
    all_corr_fisherz = np.load(f'{result_folderpath}/stim_eye_corr.npz',allow_pickle = True)['all_corr_fisherz']
else:
    all_corr = np.zeros([8,2,9])*np.nan # 8subj x 2task x 9session x 16position
if ifcalcorr:
    stim_pos = np.array([[i,j] for j in [-3,-1,1,3] for i in [-3,-1,1,3]])
    stim_ang = np.arctan2(stim_pos[:,0],stim_pos[:,1])
    for subj_i in range(len(subjs)):
        all_popt = np.load(f'{result_folderpath}/fitting_params_session.npz',allow_pickle = True)['all_popt'] # 8sub x 2task x 9sess x 16pos x 6param
        all_ang = np.arctan2(all_popt[:,:,:,:,1],all_popt[:,:,:,:,2])
        for task_i in range(2):
            for sess_i in range(9):
                eye_ang = all_ang[subj_i,task_i,sess_i,:]
                all_corr[subj_i,task_i,sess_i] = np.corrcoef(stim_ang,eye_ang)[0,1]
    all_corr_fisherz = fisherztrans(all_corr)
    np.savez(f'{result_folderpath}/stim_eye_corr.npz', all_corr=all_corr, all_corr_fisherz=all_corr_fisherz)
